# 05 Run Report And Go/No-Go
Stage-Metriken aggregieren und Gate-Entscheidung schreiben.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
import json
import pandas as pd


ROOT: /home/ubuntu/Author_Name_Disambiguation


In [2]:
from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.common.io_schema import read_parquet, validate_columns, MENTION_REQUIRED_COLUMNS
from src.common.run_report import evaluate_go_no_go, write_go_no_go_report, load_gate_config
from src.common.subset_artifacts import load_subset_mentions, resolve_manifest_paths

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
metrics_dir = RUN_DIRS["metrics"]
cluster_dir = RUN_DIRS["clusters"]
manifest_dir = RUN_DIRS["subset_manifests"]

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "05_run_consistency.json",
    extras={"notebook": "05_run_report_and_go_no_go", "latest_context_path": str(latest_context_path)},
)

gate_cfg = load_gate_config(ROOT / "configs/gates.yaml")
stage_metrics = {}

print("RUN_ID:", RUN_ID)




RUN_ID: smoke_20260213T134837Z_f0fc32b8


In [3]:
train_manifest = {}
train_manifest_path = metrics_dir / "03_train_manifest.json"
if train_manifest_path.exists():
    with train_manifest_path.open("r", encoding="utf-8") as f:
        train_manifest = json.load(f)

lspo_mentions, ads_mentions, subset_meta = load_subset_mentions(
    data_cfg=DATA,
    run_dirs=RUN_DIRS,
    run_cfg=RUN,
    run_stage=RUN_STAGE,
    allow_legacy=True,
    sampler_version="v2",
)
print(f"Loaded {subset_meta.source} subsets:")
print("  LSPO <-", subset_meta.lspo_path)
print("  ADS  <-", subset_meta.ads_path)

clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = read_parquet(clusters_path) if clusters_path.exists() else pd.DataFrame(columns=["mention_id", "block_key", "author_uid"])

schema_valid = True
try:
    validate_columns(lspo_mentions, MENTION_REQUIRED_COLUMNS, "lspo_mentions")
    validate_columns(ads_mentions, MENTION_REQUIRED_COLUMNS, "ads_mentions")
except Exception:
    schema_valid = False

manifest_paths = resolve_manifest_paths(
    run_id=RUN_ID,
    manifest_dir=manifest_dir,
    identity=subset_meta.identity,
    run_stage=RUN_STAGE,
)
determinism_valid = (
    (manifest_paths.lspo_primary.exists() and manifest_paths.ads_primary.exists())
    or (manifest_paths.lspo_legacy.exists() and manifest_paths.ads_legacy.exists())
)

consistency_files = [
    metrics_dir / "00_run_consistency.json",
    metrics_dir / "01_run_consistency.json",
    metrics_dir / "02_run_consistency.json",
    metrics_dir / "03_run_consistency.json",
    metrics_dir / "04_run_consistency.json",
    metrics_dir / "05_run_consistency.json",
]
run_id_consistent = True
for p in consistency_files:
    if not p.exists():
        run_id_consistent = False
        break
    with p.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    if str(payload.get("run_id")) != str(RUN_ID):
        run_id_consistent = False
        break

uid_uniqueness_max = int(clusters.groupby("mention_id")["author_uid"].nunique().max()) if len(clusters) else 0
uid_uniqueness_valid = uid_uniqueness_max <= 1
mention_coverage = (clusters["mention_id"].nunique() / max(1, ads_mentions["mention_id"].nunique())) if len(ads_mentions) else 0.0

lspo_pairwise_f1 = float(train_manifest["best_val_f1"]) if train_manifest.get("best_val_f1") is not None else None

stage_metrics.update({
    "run_id": RUN_ID,
    "stage": RUN_STAGE,
    "schema_valid": schema_valid,
    "determinism_valid": bool(determinism_valid),
    "uid_uniqueness_valid": bool(uid_uniqueness_valid),
    "uid_uniqueness_max": int(uid_uniqueness_max),
    "mention_coverage": float(mention_coverage),
    "run_id_consistent": bool(run_id_consistent),
    "lspo_pairwise_f1": lspo_pairwise_f1,
    "threshold": train_manifest.get("best_threshold"),
    "threshold_selection_status": train_manifest.get("best_threshold_selection_status", "unknown"),
    "threshold_source": train_manifest.get("best_threshold_source", "unknown"),
    "val_class_counts": train_manifest.get("best_val_class_counts", {}),
    "test_class_counts": train_manifest.get("best_test_class_counts", {}),
    "counts": {
        "lspo_mentions": int(len(lspo_mentions)),
        "ads_mentions": int(len(ads_mentions)),
        "ads_clusters": int(len(clusters)),
    },
})
stage_metrics



Loaded shared subsets:
  LSPO <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/lspo_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
  ADS  <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/ads_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet


{'run_id': 'smoke_20260213T134837Z_f0fc32b8',
 'stage': 'smoke',
 'schema_valid': True,
 'determinism_valid': True,
 'uid_uniqueness_valid': True,
 'uid_uniqueness_max': 1,
 'mention_coverage': 1.0,
 'run_id_consistent': True,
 'lspo_pairwise_f1': 0.9825436408977556,
 'threshold': -0.039000000000000035,
 'threshold_selection_status': 'ok',
 'threshold_source': 'val_f1_opt',
 'val_class_counts': {'pos': 198, 'neg': 10},
 'test_class_counts': {'pos': 193, 'neg': 11},
 'counts': {'lspo_mentions': 5000, 'ads_mentions': 5000, 'ads_clusters': 5000}}

In [4]:
go = evaluate_go_no_go(stage_metrics, gate_config=gate_cfg)
report_path = metrics_dir / f"05_go_no_go_{RUN_STAGE}.json"
summary_path = metrics_dir / f"05_stage_metrics_{RUN_STAGE}.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(stage_metrics, f, indent=2)
write_go_no_go_report(go, report_path)

print("GO:" if go["go"] else "NO-GO:", go["go"])
display(pd.DataFrame(go["checks"]))
if go.get("blockers"):
    print("Blockers:")
    display(pd.DataFrame({"blocker": go["blockers"]}))
else:
    print("Blockers: none")
print("stage summary:", summary_path)
print("go/no-go:", report_path)


NO-GO: False


,name,passed,detail
0,schema_valid,True,All required schemas validated
1,determinism_valid,True,Determinism checks passed
2,uid_uniqueness,True,Each mention_id mapped to one author_uid
3,run_id_consistent,True,Run ID consistent across stage artifacts
4,mention_coverage,True,"Observed coverage=1.0, required>=1.0"
5,uid_uniqueness_max,True,"Observed max=1, required<=1"
6,min_negatives_val,False,"Observed val neg=10, required>=20"
7,min_negatives_test,False,"Observed test neg=11, required>=20"
8,threshold_not_extreme,True,"Observed threshold=-0.039000000000000035, allo..."
9,threshold_selection_status,True,"status=ok, allowed=['fallback_no_labels', 'fal..."


Blockers:


,blocker
0,min_negatives_val
1,min_negatives_test


stage summary: /home/ubuntu/Author_Name_Disambiguation/artifacts/metrics/smoke_20260213T134837Z_f0fc32b8/05_stage_metrics_smoke.json
go/no-go: /home/ubuntu/Author_Name_Disambiguation/artifacts/metrics/smoke_20260213T134837Z_f0fc32b8/05_go_no_go_smoke.json
